### Imports & Functions

In [1]:
import signac
#import random
import os
import pandas as pd
#import glob
#import time
#import math
import sys
#import re
#from datetime import timedelta
import numpy as np

In [2]:
#import helper functions
path_git='../../cascade_computing' #todo - set path to git
sys.path.append(path_git+"/src")
from compute.utils import toolbox_setup as setuptools

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


### Signac project setup:

Create a new project, that contains all you simulation runs that you want to compare or evaluate together. The single simulation runs are corresponding to so-called "jobs". See the Signac doc: https://signac.readthedocs.io/en/latest/tutorial.html
For now its enough to give the project a name, and throughout this example will will add simulation runs as jobs. 

In [3]:
p_name='MUT16_MUT8' #todo - name your project

In [4]:
path=path_git+'/examples' 
project_path= path +'/' + p_name
os.makedirs(project_path, exist_ok=True)

In [5]:
project_path

'../../cascade_computing/examples/MUT16_MUT8'

### metadata: system composition

specify any molecules in you system  - that helps in labeling and analysis your analysis result

In [6]:
#single "full length pdb" files
path_input=path_git+'/examples/data_MUT16_MUT8/components_full_length'

#all system incredients
filenameA='MUT16'
filenameB='MUT8'

names={'A':'MUT16', 'B':'MUT8'} #todo check chain names in top/pdb files

if your molecules have one or more domains (e.g. DOM) introduce columns like DOM_min, DOM_max

In [7]:
#no domains
domains=['NTERM', 'FULL']

mut16 = {
    'prot': "MUT16",
    'orig_na': 2204,
    'cut_na': 140,
    'min': 633,
    'max': 772,
    'NTERM_MIN':633,
    'NTERM_MAX':700,
    'FF_MIN':701,
    'FF_MAX':772,
    'file': f"{path_input}/{filenameA}.pdb"
}

mut8 = {
    'prot': "MUT8",
    'orig_na': 805,
    'cut_na': 51,
    'min': 1,
    'max': 51,
    'NTERM_MIN':0,
    'NTERM_MAX':0,
    'FF_MIN':1,
    'FF_MAX':20,
    'file': f"{path_input}/{filenameB}.pdb"
}

# Combine into a list and create the DataFrame
domain_list = [mut16, mut8]
df_domains = pd.DataFrame(domain_list)

In [8]:
df_domains.file.values

array(['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb',
       '../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT8.pdb'],
      dtype=object)

In [9]:
#add sequences: from the full length pdb files the sequence you're simulating is extracted based on the min/max values
df_domains=setuptools.add_sequences(df_domains)
df_domains

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


,prot,orig_na,cut_na,min,max,NTERM_MIN,NTERM_MAX,FF_MIN,FF_MAX,file,full_seq,cut_seq
0,MUT16,2204,140,633,772,633,700,701,772,../../cascade_computing/examples/data_MUT16_MU...,MSESDDDYPELDISDQYIDPLGIVVGPPPASYTETDREETPMQNRT...,VPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPK...
1,MUT8,805,51,1,51,0,0,1,20,../../cascade_computing/examples/data_MUT16_MU...,MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNK...,MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNK...


### project data: constant system parameters 

Specify your simulation parameters, it will help the analysis & documentation. 
Only specify parameters here that will be the same for all jobs/datasets in the project.

In [10]:
#meta-data (also for analysis and documentation)

#box size
lx=15
ly=15
lz=30
d=0


#minimization & equilibration (for documentation)
min_steps=20000 #stepsize see .mdp
eq_steps=50000 #stepsize see .mdp
dt= 0.02 #ps timestep see .mdp

#simulations time 
t_tot= 0.1 #in microseconds
t_ps=t_tot* 1e6 
steps=int(t_ps/dt) #overall simulation steps


times={'t_ps':t_ps, 'dt':dt,'steps':steps, 'min_steps':min_steps, 'eq_steps':eq_steps}

In [11]:
#system composition

#simulation data path (output trajectories from simulation)
data_path=path_git+'/examples/data_MUT16_MUT8' 

#protein system
top_mix=data_path+'/topol.top'

#protein
top_org=data_path+'/dynamics.gro'


#solvated system
top=data_path+'/dynamics_fah.tpr'

#trajectory path
traj_dir=data_path

#optional: gather xtc files from a location
#specify list of trajectory files 
#l_xtc=['/lustre/miifs01/project/m2_trr146/lubaltz/gmx_output/KG_data/replica_1/dynamics/dynamics_skip100.xtc']

In [12]:
#structures
n_starting_structures=1 #number of different structures (system topologies)
l_struc_inital=np.arange(n_starting_structures)
l_struc_inital

array([0])

### project data: sweep system parameters 

Make lists or define sets of the parameter that you want to sweep through. Then one job in the project can have a unique parameter value (e.g. a distinct replica, a different composition or different box size)

In [13]:
#e.g lists to iterate over different structures
l_filenames=[[filenameA, filenameB]] 

#e.g lists to iterate over different numbers of chain A, chain B in the system
l_n_mols=[[100,10]] #molnumbers

In [14]:
#replicas
n_replicas=2 #per jobs in project setup
#l_replicas=np.arange(n_replicas)+1
l_replicas=[0,1]

In [15]:
#different structures/system topology
#(here constant - defined above)
#l_struc_inital=np.arange(n_starting_structures)

### project data: system composition table

create a table (df_sys_domains) as a system overview containing all molecules, domains and their ids

In [16]:
#structure - here constant for all jobs (otherwise include into loop)
filenames = l_filenames[0]
molA      = path_input+"/"+filenames[0]
molB      = path_input+"/"+filenames[1]

#labels and files [mol_a, mol_b]
geo_mols  = [molA+'.pdb', molB+'.pdb']
labels    = ["A", "B"] #label in the simulations setup (keep it)
mols      = [filenameA,filenameB]

#number of chains [n_a, n_b]
n_mols    = l_n_mols[0]

In [17]:
sys={'path_input':path_input, 'filenameA':filenames[0],'filenameB': filenames[1],'molA':molA, 'molB':molB, 'names':names}
print('sys',sys)

sys {'path_input': '../../cascade_computing/examples/data_MUT16_MUT8/components_full_length', 'filenameA': 'MUT16', 'filenameB': 'MUT8', 'molA': '../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16', 'molB': '../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT8', 'names': {'A': 'MUT16', 'B': 'MUT8'}}


In [18]:
#tofix
##setuptools.calc_sys_domain(molA, molB, df_domains, n_mols)

In [19]:
#create system domain dataframe
files=[[item, lab] for count, item, lab in zip(n_mols, geo_mols, labels) for _ in range(count)]
print("files", files)
df_sys_domains=setuptools.make_sys_mon_df(files,df_domains)
df_sys_domains=setuptools.make_sys_mon_df_max(df_sys_domains,df_domains)
df_sys_domains=setuptools.make_sys_mon_df_max_domains(df_sys_domains,df_domains,domains)

stuc_id_A=df_sys_domains[df_sys_domains["unit"]=="A"]["struc_id"].min()
stuc_id_B=df_sys_domains[df_sys_domains["unit"]=="B"]["struc_id"].min()

struc_A=df_sys_domains[df_sys_domains["struc_id"]==stuc_id_A]["prot"].values
struc_B=df_sys_domains[df_sys_domains["struc_id"]==stuc_id_B]["prot"].values
mols=list([list(struc_A), list(struc_B)])

files [['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/MUT16.pdb', 'A'], ['../../cascade_computing/examples/data_MUT16_MUT8/components_full_length

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


seg <Segment A>
l seg 1054
monomer_unit 0 VPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQ
chain 0 MSESDDDYPELDISDQYIDPLGIVVGPPPASYTETDREETPMQNRTEDDTNSYGNSSGEHDYDSYLDSGDDDFDVDAYYANDNMDEPPPETIPDNLIQNVIGRNDNADYDFSDASNPEIMKRDLQLFLSSSLLSNPLKFRSGYSSDELDDCLKSCMGYSLQVTAVLLLPPEIISQLPNDSKTEHAHALVRGGWLQSKEGAFFPVISDSERETVVSLMNGSEEQHKRQERKKKEADTFESEEKEIRTLLTFNMIAELLMAVRNEYSIRSVKYQILSTAYTNMVTGAAHANIFRKYKDILQLDPEKLWNNDWFKEYTNRGTLKKFLTTARFSEIVVSQANGKTVELYFRADDEGNRPVVLFTDEHIADVRNKWKTGNQRNQNYGSQGNYRAGGQRSDDRRGPQQRRNVIVPDPNYQPSTFAGGISNNADDDGSLQPTTSSHFNRNTDRSTSRPPRAPTSPVNRVMETDPLMGQGTSSGAPQRSAIPNPFGGAPALSRSTITNGNRGPSYGDRGERVQDVGDTTSDSEITSEGSYSDEDPEQKEIKRQRRKDKLKKKQERELRSREKHTKSKQQPPSKIETRFNTYKKKSESSATDTSNTPPVDTVNVALPTPVVESSSTTAAPSIPVSTRPEVVVPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQQYPPVNQQQPIYQQPAPQYPPYNSIQNNPQHGPSPFN

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


seg <Segment A>
l seg 1054
monomer_unit 0 VPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQ
chain 0 MSESDDDYPELDISDQYIDPLGIVVGPPPASYTETDREETPMQNRTEDDTNSYGNSSGEHDYDSYLDSGDDDFDVDAYYANDNMDEPPPETIPDNLIQNVIGRNDNADYDFSDASNPEIMKRDLQLFLSSSLLSNPLKFRSGYSSDELDDCLKSCMGYSLQVTAVLLLPPEIISQLPNDSKTEHAHALVRGGWLQSKEGAFFPVISDSERETVVSLMNGSEEQHKRQERKKKEADTFESEEKEIRTLLTFNMIAELLMAVRNEYSIRSVKYQILSTAYTNMVTGAAHANIFRKYKDILQLDPEKLWNNDWFKEYTNRGTLKKFLTTARFSEIVVSQANGKTVELYFRADDEGNRPVVLFTDEHIADVRNKWKTGNQRNQNYGSQGNYRAGGQRSDDRRGPQQRRNVIVPDPNYQPSTFAGGISNNADDDGSLQPTTSSHFNRNTDRSTSRPPRAPTSPVNRVMETDPLMGQGTSSGAPQRSAIPNPFGGAPALSRSTITNGNRGPSYGDRGERVQDVGDTTSDSEITSEGSYSDEDPEQKEIKRQRRKDKLKKKQERELRSREKHTKSKQQPPSKIETRFNTYKKKSESSATDTSNTPPVDTVNVALPTPVVESSSTTAAPSIPVSTRPEVVVPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQQYPPVNQQQPIYQQPAPQYPPYNSIQNNPQHGPSPFN

...

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


seg <Segment A>
l seg 570
monomer_unit 0 VPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQ
chain 0 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRNYLFYTKNIDGFVEISGNIQSLQIGRWLKLTVCQNFVGQYVLDSSNSCGYENWDVLPNHRNSSARFFPKYEANDEGEILITARFDVIVDRKNNLYDFTSYDISYDHIIDDFSLIKSSCQHYDGRSFIVEAQASLAGWVALSVHEDPQKLHTCDALPNPVVDSDVQYLHDTRLLDIMTGNGYVQTPETDVQDQQSSQHQEDVHSQMNSQTSDSYNSSRVVSENREIPPETFRSQSIELNQVDSSLSQTTISSRAAPVIDSNQLSDEDEIDDEDTYGTRGTSNIPMRPFIKDLAPTMLQLLRQDKTDSEKPQSALCTVVQKIDGFAILYTAKRDVINVLLQERSCEGLERSPQLGDVAFFDILPRRIETKDRLIFKIPYTHIAVKKKPDTPDSLLKIDCFKNSVRCFGGVLEMKVKIALSKPELVVEQYHDNTEMNSDHHFYYLKATNGVLVTIPKERLLNHLNSKLSADFDLIAWVVHRKPIGNVSLHIGKGGEAYQQFTNGDIRELPPLSSNQYFMNVRK
monomer_unit 1 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRN
chain 1 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRNYLFYTKNIDGFVEISGNIQSLQIGRWLKLTVCQNFVGQYVLDSSNSCGYENWDVLPNHRNSSARFFPKYEANDEGEILITARFDVIVDRKNNLYDFTSYDISYDHIIDDFSLI

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/coordinates/PDB.py:453: UserWarning: 1 A^3 CRYST1 record, this is usually a placeholder. Unit cell dimensions will be set to None.
  warnings.warn("1 A^3 CRYST1 record,"


seg <Segment A>
l seg 570
monomer_unit 0 VPPENPAPLREVGNFYSKSNHDEDRRNVQLPFTPADTHKPIKVAPKEPVRNPLLKERPSANGFINRRLPSHPAPPPVNQSQPANQPMQTAVYQNSHPGAPYIPQQPTYQPQLPVQQPQPHQYAPQPIHHQQPIHQPMHGQ
chain 0 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRNYLFYTKNIDGFVEISGNIQSLQIGRWLKLTVCQNFVGQYVLDSSNSCGYENWDVLPNHRNSSARFFPKYEANDEGEILITARFDVIVDRKNNLYDFTSYDISYDHIIDDFSLIKSSCQHYDGRSFIVEAQASLAGWVALSVHEDPQKLHTCDALPNPVVDSDVQYLHDTRLLDIMTGNGYVQTPETDVQDQQSSQHQEDVHSQMNSQTSDSYNSSRVVSENREIPPETFRSQSIELNQVDSSLSQTTISSRAAPVIDSNQLSDEDEIDDEDTYGTRGTSNIPMRPFIKDLAPTMLQLLRQDKTDSEKPQSALCTVVQKIDGFAILYTAKRDVINVLLQERSCEGLERSPQLGDVAFFDILPRRIETKDRLIFKIPYTHIAVKKKPDTPDSLLKIDCFKNSVRCFGGVLEMKVKIALSKPELVVEQYHDNTEMNSDHHFYYLKATNGVLVTIPKERLLNHLNSKLSADFDLIAWVVHRKPIGNVSLHIGKGGEAYQQFTNGDIRELPPLSSNQYFMNVRK
monomer_unit 1 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRN
chain 1 MHNGYHSYFPPNHYYAQSQPSSSYNPQPQIQQPQTQIYGLVVGYNKYGNRNYLFYTKNIDGFVEISGNIQSLQIGRWLKLTVCQNFVGQYVLDSSNSCGYENWDVLPNHRNSSARFFPKYEANDEGEILITARFDVIVDRKNNLYDFTSYDISYDHIIDDFSLI

/fshpc/lubaltz/code/cascade_computing/examples/../../cascade_computing/src/compute/utils/toolbox_setup.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sys_mon_df[col]= sys_mon_df[col].fillna(0).astype(int)


In [20]:
df_sys_domains

,prot,struc_id,unit,min,max
0,MUT16,0,A,1,140
1,MUT16,1,A,141,280
2,MUT16,2,A,281,420
3,MUT16,3,A,421,560
4,MUT16,4,A,561,700
...,...,...,...,...,...
105,MUT8,105,B,14256,14306
106,MUT8,106,B,14307,14357
107,MUT8,107,B,14358,14408
108,MUT8,108,B,14409,14459


## PROJECT - RUN SETUP

In [21]:
#create project
project = signac.init_project(project_path) #create new
#project = signac.get_project(project_path) #add to excisting

In [25]:
setuptools.gather_xtcs(traj_dir,bool_pp=False, key="frame", part=False)

[]

In [26]:
traj_dir

'../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/CLONE_1'

In [27]:
! ls ../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/CLONE_1 

ls: cannot access '../../cascade_computing/examples/data_MUT16_MUT8/components_full_length/CLONE_1': No such file or directory


In [30]:
#create state_points & create notebooks accordingly

for struc_init in l_struc_inital:
    for replica in l_replicas:

       #simulation data
        
        #protein system
        #top_mix (constant - defined above)
        
        #protein
        #top_org (constant - defined above)
        

        #trajectory path
        traj_dir=f"{data_path}/CLONE_{replica}"

        
        #solvated system
        #top=traj_dir+'/dynamics.tpr' (constant - defined above)
        
        
        #list of trajectory files
        l_xtc=[setuptools.gather_xtcs(traj_dir,bool_pp=False, key="frame", part=False)]

        #signac state point
        sp ={'mols':mols, 'n_mols':n_mols, 'geo_mols':geo_mols, "replica_index":replica, "structure_index":struc_init}
        job = project.open_job(sp).init()

        #tag - "run"
        sp_str = p_name+'_' +'_'.join(f"{'-'.join(mol)}_{n}" for mol, n in zip(mols, n_mols) if n > 0) + f"_{job}"
        dict_in={'sgnc_path': job.path,'run':sp_str}

        #save meta data
        job.data['df_domains']=df_domains
        job.document['domains']=domains
        job.data['df_sys_domains']=df_sys_domains
        

        #documentation
        job.document['output_path']=data_path+f"/CLONE_{replica}"
        job.document['top']=top
        job.document['top_mix']=top_mix
        job.document['top_org']=top_org
        job.document['xtc']=l_xtc
        
        
        print(dict_in)
        job.document['params']=dict_in

{'sgnc_path': '/fshpc/lubaltz/code/cascade_computing/examples/MUT16_MUT8/workspace/afbfced82b24a75578176468f86e0752', 'run': 'MUT16_MUT8_MUT16_100_MUT8_10_afbfced82b24a75578176468f86e0752'}
{'sgnc_path': '/fshpc/lubaltz/code/cascade_computing/examples/MUT16_MUT8/workspace/6d191176b8db4ed51cd80d1f755027a0', 'run': 'MUT16_MUT8_MUT16_100_MUT8_10_6d191176b8db4ed51cd80d1f755027a0'}


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/synced_collections/numpy_utils.py:46: NumpyConversionWarning: Any numpy types provided will be transparently converted to the closest base Python equivalents.
  warnings.warn(NUMPY_CONVERSION_WARNING, NumpyConversionWarning)
/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/synced_collections/numpy_utils.py:46: NumpyConversionWarning: Any numpy types provided will be transparently converted to the closest base Python equivalents.
  warnings.warn(NUMPY_CONVERSION_WARNING, NumpyConversionWarning)


## PROJECT - CHECK SETUP

In [29]:
project

,sp.mols,sp.n_mols,sp.geo_mols,sp.replica_index,sp.structure_index,doc.domains,doc.output_path,doc.top,doc.top_mix,doc.top_org,doc.xtc,doc.params
afbfced82b24a75578176468f86e0752,"[[MUT16], [MUT8]]","[100, 10]",[../../cascade_computing/examples/data_MUT16_M...,0,0,"[NTERM, FULL]",../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,[[../../cascade_computing/examples/data_MUT16_...,{'sgnc_path': '/fshpc/lubaltz/code/cascade_com...
6d191176b8db4ed51cd80d1f755027a0,"[[MUT16], [MUT8]]","[100, 10]",[../../cascade_computing/examples/data_MUT16_M...,1,0,"[NTERM, FULL]",../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,[[../../cascade_computing/examples/data_MUT16_...,{'sgnc_path': '/fshpc/lubaltz/code/cascade_com...


In [24]:
#check upon project
print(project.path)
print(project.workspace)
df = project.to_dataframe()
df

/fshpc/lubaltz/code/cascade_computing/examples/MUT16_MUT8
/fshpc/lubaltz/code/cascade_computing/examples/MUT16_MUT8/workspace


,sp.mols,sp.n_mols,sp.geo_mols,sp.replica_index,sp.structure_index,doc.domains,doc.output_path,doc.top,doc.top_mix,doc.top_org,doc.xtc,doc.params
afbfced82b24a75578176468f86e0752,"[[MUT16], [MUT8]]","[100, 10]",[../../cascade_computing/examples/data_MUT16_M...,0,0,"[NTERM, FULL]",../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,[[]],{'sgnc_path': '/fshpc/lubaltz/code/cascade_com...
6d191176b8db4ed51cd80d1f755027a0,"[[MUT16], [MUT8]]","[100, 10]",[../../cascade_computing/examples/data_MUT16_M...,1,0,"[NTERM, FULL]",../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,../../cascade_computing/examples/data_MUT16_MU...,[[]],{'sgnc_path': '/fshpc/lubaltz/code/cascade_com...


In [25]:
#choose a single job
job = project.open_job(id='afbfced82b24a75578176468f86e0752')

In [27]:
for job in project:
    print(job.doc.output_path)

/home/lubaltz/code/cascade_computing/examples/data/replica_6
/home/lubaltz/code/cascade_computing/examples/data/replica_10
